# Circuit compilation: DAG passes, optimization levels, and routing

qarpx compiles every circuit through the same pipeline before it reaches a simulator:

```
Block tree ──flatten()──▶ flat commands
    ──transpile (rebase to the target gate set)──▶
    ──CircuitDAG passes (cancel / merge, level-dependent)──▶
    ──single-qubit fusion──▶
    ──[device path: route to the architecture]──▶  simulator
```

All of this runs **automatically** inside the engines (`QarpEngine`) — you
never have to call it. This notebook shows how it all works: the `Block.optimize()` levels, the
`qx.CircuitDAG` wire-dependency graph behind them, the two routers (`Sabre`, `Lite`), and
the phase-exactness guarantee the pipeline follows.

Contracts live in `docs/contracts/qarp_conventions.md` §16.

In [ ]:
import numpy as np
import qarpx as qx

from qarp.blocks import SimpleBlock
from qarp.devices import Architecture, Device, compile_for_device

## 1. Optimization levels

`Block.optimize(level=...)` lowers to a gate set and then simplifies on the circuit DAG:

| level | what runs |
|---|---|
| 0 | transpile only |
| 1 | + **wire-adjacent** cancellation/merging + single-qubit fusion *(engine default)* |
| 2 | + **commutation-aware** cancellation (gates combine across provably-commuting gates) |

The circuit below hides three distinct simplifications: an `H` pair separated by a gate on
*another* qubit (wire-adjacent — a textual scan would miss it), a zero-angle rotation, and
an `Rz` pair separated by a `CX` sharing only its **control** (needs commutation analysis:
`Rz` and the control are both Z-diagonal).

In [ ]:
block = SimpleBlock(3, name="redundant")
block.h(0)
block.x(1)          # sits between the H pair, but on another qubit
block.h(0)          # ← cancels with the first H at level >= 1
block.rz(1, qx.Param(0.0))   # ← zero rotation, dropped at level >= 1
block.rz(2, qx.Param(0.3))
block.cx(2, 0)      # Rz commutes through the CX *control* (both Z-diagonal)
block.rz(2, qx.Param(0.4))   # ← merges with the first Rz at level 2
block.build()

for level in (0, 1, 2):
    opt = block.optimize(level=level)
    gates = [qx.gate_name(c.gate) for c in opt.flatten()]
    print(f"level {level}: {len(gates)} commands, depth {opt.depth()}, "
          f"{opt.n_1q_gates()} 1q + {opt.n_2q_gates()} 2q gates  {gates}")

Every level preserves the unitary **exactly** — global phase included. That invariant
(`EQ-2` in the conventions doc) is what lets the engines apply level 1 unconditionally.

In [ ]:
u_ref = block.unitary_matrix()
for level in (0, 1, 2):
    diff = np.linalg.norm(u_ref - block.optimize(level=level).unitary_matrix())
    print(f"level {level}: ||U_ref - U_opt|| = {diff:.2e}")

## 2. The circuit DAG

The passes run on `qx.CircuitDAG`: commands become nodes, and edges are per-wire ordering
dependencies (qubit wires, classical-bit wires, one global-phase wire). Two gates on
disjoint qubits have no edge between them — that is how the `H` pair above was "adjacent"
despite the `X` sitting between them textually.

The DAG is exposed read-only for introspection; `to_commands()` round-trips **exactly**
(a stable topological sort that reproduces the input order — contract `RT-1`).

In [ ]:
cmds = block.flatten()
dag = qx.CircuitDAG.from_commands(cmds)

print("nodes:", dag.n_nodes, "| depth:", dag.depth())
print("ops:  ", dag.count_ops())
print("ASAP layers (node ids):", dag.layers())
print("front layer (ready at t=0):", dag.front_layer())

round_trip = dag.to_commands()
print("round-trip exact:", all(a == b for a, b in zip(round_trip, cmds)))

`depth()` is the longest dependency path — gates on disjoint qubits share a time step
(`Barrier`/`GPhase` weigh 0). It is also available directly as `block.depth()`.

Gate-count resource metrics are available directly on the block too: `n_1q_gates()` /
`n_2q_gates()` (shorthands for the general `n_nqb_gates(k)`, excluding non-gate commands
like `Barrier`/`Measure`/`Reset`/`GPhase`/branch markers), and the unfiltered
`n_gates_of_type(gate)` for a specific `qx.GateType` (mirrors `dag.count_ops()` above).

In [ ]:
print("n_1q_gates:", block.n_1q_gates(), "| n_2q_gates:", block.n_2q_gates())
print("n_gates_of_type(H): ", block.n_gates_of_type(qx.GateType.H))
print("n_gates_of_type(Rz):", block.n_gates_of_type(qx.GateType.Rz))
print("n_gates_of_type(CX):", block.n_gates_of_type(qx.GateType.CX))

## 3. A real workload

On structured circuits the win compounds: Trotter layers are full of rotation merges and
inverse pairs. (This is exactly what `QarpEngine` does to every circuit at build time.)

In [ ]:
from qarp.operators import QubitOperator

from qarp.blocks import TrotterBlock

ham = (QubitOperator("X0 X1", 0.5) + QubitOperator("Y0 Y1", 0.5)
       + QubitOperator("Z0 Z1", 0.6) + QubitOperator("Z0", 0.3)
       + QubitOperator("Z1", 0.3))
trotter = TrotterBlock(operator=ham, n_qubits=2, steps=3, time=0.8, order=2).build()

n0 = len(trotter.flatten())
for level in (1, 2):
    opt = trotter.optimize(level=level)
    n = len(opt.flatten())
    print(f"level {level}: {n0} -> {n} commands "
          f"({100 * (1 - n / n0):.0f}% fewer), depth {opt.depth()}")

diff = np.linalg.norm(trotter.unitary_matrix() - trotter.optimize(level=2).unitary_matrix())
print(f"unitary preserved: ||dU|| = {diff:.2e}")

## 4. Routing: Sabre (default) vs Lite

Real hardware only couples neighbouring qubits, so distant 2-qubit gates need `SWAP`s.
qarpx ships two routers behind one contract:

- **`Lite`** — greedy shortest-path sweep: for each unsatisfiable gate, SWAP one hop at a
  time toward the partner.
- **`Sabre`** *(default)* — schedules on the DAG's front layer with lookahead, searches for
  a good **initial qubit placement** (forward–reverse–forward traversal), and carries a
  portfolio guarantee: it is never worse than Lite (benchmark: 42% fewer SWAPs overall).

The starkest case: one long-range gate on a line. Lite must ferry the qubits together;
Sabre just *starts* them adjacent.

In [ ]:
line8 = Architecture(8, [(i, i + 1) for i in range(7)], "line-8")

circuit = SimpleBlock(8, name="distant_pair")
circuit.h(0)
circuit.cx(0, 7)          # qubits 0 and 7 sit at opposite ends of the line
circuit.rz(7, qx.Param(0.4))
circuit.build()

def swap_count(kind):
    opts = qx.RoutingOptions()
    opts.arch = line8
    opts.router = kind
    routed = qx.route(circuit.flatten(), opts)
    return sum(1 for c in routed.commands if c.gate == qx.GateType.SWAP), routed

lite_swaps, lite_res = swap_count(qx.RouterKind.Lite)
sabre_swaps, sabre_res = swap_count(qx.RouterKind.Sabre)
print(f"Lite:  {lite_swaps} SWAPs   (identity placement, ferry across the line)")
print(f"Sabre: {sabre_swaps} SWAPs   (initial mapping places the pair adjacent)")
print("Lite  initial -> final:", list(lite_res.initial_logical_to_physical), "->", list(lite_res.final_logical_to_physical))
print("Sabre initial -> final:", list(sabre_res.initial_logical_to_physical), "->", list(sabre_res.final_logical_to_physical))

The routed commands act on **physical** qubits, and every routed result carries two maps:
`initial_logical_to_physical` says which wire each logical qubit is *placed* on before the
first gate (Lite: the identity; Sabre: its search's choice), and `final_logical_to_physical`
says where it *ends* after the inserted SWAPs. Sampled counts are reindexed by the final map
(`qx.reindex_sampling_result` — engines apply it automatically); anything that prepares a
state *before* the circuit, such as an injected `initial_state`, must respect the initial one.

`compile_for_device` bundles the full device pipeline — rebase → route → re-rebase. The
second rebase pass decomposes router-introduced SWAPs whenever the device gate set lacks
them (here `qulacs_gateset` *contains* SWAP, so Lite's six survive as-is):

In [ ]:
device = Device(8, architecture=line8, gate_set=qx.qulacs_gateset())

out = compile_for_device(circuit, device)                            # Sabre default
out_lite = compile_for_device(circuit, device, router=qx.RouterKind.Lite)

def profile(compiled):
    ops = {}
    for c in compiled.commands:
        ops[qx.gate_name(c.gate)] = ops.get(qx.gate_name(c.gate), 0) + 1
    return ops

print("Sabre:", profile(out))
print("Lite: ", profile(out_lite))

## 5. Phase exactness

Rebasing `P`/`U`/`CP`/`CU` away emits a compensating `GPhase`, so even the *global* phase
of the unitary survives compilation. That matters the moment a compiled circuit is used as
a **sub**-circuit — controlled, in QPE, QSVT, or LCU — where a global phase becomes a
relative (measurable) one. The level-1 pass then merges all `GPhase` commands into one.

In [ ]:
phased = SimpleBlock(2, name="phase_family")
phased.p(0, qx.Param(0.7))
phased.cp(0, 1, qx.Param(0.5))
phased.u(1, qx.Param(0.3), qx.Param(0.2), qx.Param(0.1))
phased.build()

# qulacs_gateset has no P/CP/U -> all three decompose.
rebased = phased.optimize(target_gateset=qx.qulacs_gateset(), level=1)

diff = np.linalg.norm(phased.unitary_matrix() - rebased.unitary_matrix())
gphases = sum(1 for c in rebased.flatten() if c.gate == qx.GateType.GPhase)
print(f"||U - U_rebased|| = {diff:.2e}   (exact, global phase included)")
print(f"compensating GPhase commands after the level-1 merge: {gphases}")

## 6. Where this happens automatically

You rarely call any of the above directly:

- **`QarpEngine`** runs transpile + level-1 DAG optimization on every
  circuit at `build()` time, and the full rebase → route → re-rebase pipeline whenever the
  engine has a `Device` with an architecture (Sabre routing by default).
- **`Block.optimize(target_gateset, level)`** is the manual entry point shown here —
  useful before inspecting, plotting, or exporting a circuit.
- **`qx.CircuitDAG`** is the read-only window into the dependency structure
  (`depth`, `layers`, `count_ops`, `front_layer`).

Further reading: `docs/contracts/qarp_conventions.md` §16 (contracts).